In [1]:
!pip install ultralytics roboflow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 31.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 249.2/249.2 kB 30.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 89.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 147.2 MB/s eta 0:00:00
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 4.13.0.92
    Uninstalling opencv-python-headless-4.13.0.92:
      Successfully uninstalled opencv-python-headless-4.13.0.92
  Attempting uninstall: idna
    Found existing installation: idna 3.15
    Uninstalling idna-3.15:
      Successfully uninstalled idna-3.15


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
from roboflow import Roboflow

rf = Roboflow(api_key="64g4PSrg0kTCRLoALd8j")

# Best option: 3,489 images — largest seatbelt dataset on Roboflow
project = rf.workspace("seatbelttraining-7yh0f").project("seatbelt-detection-lb1ec")
dataset = project.version(3).download("yolov11")

DATASET_DIR = dataset.location
print("Dataset downloaded to:", DATASET_DIR)

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to seatbelt-detection-3 in yolov11:: 100%|██████████| 9387/9387 [00:01<00:00, 7310.70it/s]

Dataset downloaded to: /content/seatbelt-detection-3


In [4]:
import torch

MODEL_ARCH        = 'yolo11n.pt'          # pretrained, NOT .yaml
EPOCHS            = 100
BATCH_SIZE        = 16
IMG_SIZE          = 640
CONFIDENCE_THRESHOLD = 0.25
PROJECT_NAME      = '/content/drive/MyDrive/YOLO11-Seatbelt'  # saves to Drive
EXPERIMENT_NAME   = 'exp1'
DATA_CONFIG       = f'{DATASET_DIR}/data.yaml'  # Roboflow generates this automatically

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")
print(f"Data config: {DATA_CONFIG}")

Using device: cuda
Data config: /content/seatbelt-detection-3/data.yaml


In [5]:
from ultralytics import YOLO

model = YOLO(MODEL_ARCH)

train_results = model.train(
    data=DATA_CONFIG,
    epochs=EPOCHS,
    batch=BATCH_SIZE,
    imgsz=IMG_SIZE,
    project=PROJECT_NAME,
    name=EXPERIMENT_NAME,
    device=device,
    exist_ok=True,
    patience=15,
    optimizer='SGD',
    lr0=0.001,
    lrf=0.01,
)

print("\nTraining completed!\n")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.60 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/seatbelt-detection-3/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v

In [6]:
best_weights_path = f'{PROJECT_NAME}/{EXPERIMENT_NAME}/weights/best.pt'
model = YOLO(best_weights_path)

validation_results = model.val(
    data=DATA_CONFIG,
    split='val',
    conf=CONFIDENCE_THRESHOLD,
)

metrics = validation_results.box
precision = getattr(metrics, 'mp', None)
recall    = getattr(metrics, 'mr', None)
map50     = getattr(metrics, 'map50', None)

f1 = 2 * (precision * recall) / (precision + recall) if precision and recall else None

print(f"Precision : {precision:.2f}")
print(f"Recall    : {recall:.2f}")
print(f"mAP50     : {map50:.2f}")
print(f"F1 Score  : {f1:.2f}")

Ultralytics 8.4.60 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO11n summary (fused): 101 layers, 2,582,542 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 883.2±285.5 MB/s, size: 21.5 KB)
val: Scanning /content/seatbelt-detection-3/valid/labels.cache... 390 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 390/390 109.1Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 25/25 4.3it/s 5.9s
                   all        390        392       0.98       0.89      0.904      0.634
           no-seatbelt         57         57      0.978      0.786      0.822      0.518
              seatbelt        333        335      0.982      0.994      0.986       0.75
Speed: 2.9ms preprocess, 5.0ms inference, 0.0ms loss, 1.5ms postprocess per image
Results saved to /content/runs/detect/val
Precision : 0.98
Recall    : 0.89
mAP50     : 0.90
F1 Score  : 0.93
